In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HotelBigDataSQL") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession khởi tạo thành công Spark version: {spark.version}")

SparkSession khởi tạo thành công Spark version: 4.1.1


In [5]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("hdfs://localhost:9000/user/hotel/data/data_cleaned_hotel.csv")

print(f"Load thành công!")
print(f"Tổng số dòng : {df.count():,}")
print(f"Số cột       : {len(df.columns)}")

Load thành công!
Tổng số dòng : 119,390
Số cột       : 32


In [8]:
print("SCHEMA CỦA DATASET")
df.printSchema()



SCHEMA CỦA DATASET
root
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: integer (nullable = true)
 |-- arrival_date_day_of_month: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: integer (nullable = true)
 |-- babies: integer (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_

In [9]:
df.createOrReplaceTempView("hotel")
print("Đã có temp view 'hotel")

Đã có temp view 'hotel


In [10]:
result_1 = spark.sql("""
    SELECT
        hotel,
        arrival_date_month,
        COUNT(*)                                          AS total_bookings,
        SUM(is_canceled)                                  AS total_canceled,
        ROUND(SUM(is_canceled) / COUNT(*) * 100, 2)       AS cancel_rate_pct
    FROM hotel
    GROUP BY
        hotel,
        arrival_date_month
    ORDER BY
        hotel,
        total_bookings DESC
""")

print("CÂU 1: PHÂN TÍCH MÙA VỤ ĐẶT PHÒNG")
result_1.show(30, truncate=False)

CÂU 1: PHÂN TÍCH MÙA VỤ ĐẶT PHÒNG
+------------+------------------+--------------+--------------+---------------+
|hotel       |arrival_date_month|total_bookings|total_canceled|cancel_rate_pct|
+------------+------------------+--------------+--------------+---------------+
|City Hotel  |August            |8983          |3602          |40.1           |
|City Hotel  |May               |8232          |3653          |44.38          |
|City Hotel  |July              |8088          |3306          |40.88          |
|City Hotel  |June              |7894          |3528          |44.69          |
|City Hotel  |October           |7605          |3268          |42.97          |
|City Hotel  |April             |7480          |3465          |46.32          |
|City Hotel  |September         |7400          |3110          |42.03          |
|City Hotel  |March             |6458          |2386          |36.95          |
|City Hotel  |February          |4965          |1901          |38.29          |
|City 

In [11]:
result_2 = spark.sql("""
    SELECT
        reserved_room_type,
        COUNT(*)                                                                        AS canceled_bookings,
        ROUND(SUM(adr * (stays_in_weekend_nights + stays_in_week_nights)), 2)           AS revenue_lost,
        DENSE_RANK() OVER (
            ORDER BY SUM(adr * (stays_in_weekend_nights + stays_in_week_nights)) DESC
        )                                                                               AS loss_rank
    FROM hotel
    WHERE is_canceled = 1
    GROUP BY reserved_room_type
    ORDER BY loss_rank
""")

print("CÂU 2: XẾP HẠNG THIỆT HẠI TÀI CHÍNH THEO LOẠI PHÒNG")
result_2.show(truncate=False)

CÂU 2: XẾP HẠNG THIỆT HẠI TÀI CHÍNH THEO LOẠI PHÒNG
+------------------+-----------------+-------------+---------+
|reserved_room_type|canceled_bookings|revenue_lost |loss_rank|
+------------------+-----------------+-------------+---------+
|A                 |33630            |1.010502932E7|1        |
|D                 |6102             |3277414.76   |2        |
|E                 |1914             |1270682.94   |3        |
|G                 |763              |741340.95    |4        |
|F                 |880              |670647.29    |5        |
|C                 |308              |287410.82    |6        |
|H                 |245              |241167.78    |7        |
|B                 |368              |133399.26    |8        |
|L                 |2                |144.0        |9        |
|P                 |12               |0.0          |10       |
+------------------+-----------------+-------------+---------+



In [12]:
result_3 = spark.sql("""
    SELECT
        CASE
            WHEN is_repeated_guest = 1 THEN 'Khach quen'
            WHEN is_repeated_guest = 0 THEN 'Khach moi'
        END                                                 AS guest_type,
        COUNT(*)                                            AS total_bookings,
        SUM(is_canceled)                                    AS total_canceled,
        ROUND(SUM(is_canceled) / COUNT(*) * 100, 2)         AS cancel_rate_pct
    FROM hotel
    GROUP BY
        CASE
            WHEN is_repeated_guest = 1 THEN 'Khach quen'
            WHEN is_repeated_guest = 0 THEN 'Khach moi'
        END
    ORDER BY total_bookings DESC
""")

print("CÂU 3: SO SÁNH HÀNH VI KHÁCH QEN VÀ KHÁCH MỚI")
result_3.show(truncate=False)

CÂU 3: SO SÁNH HÀNH VI KHÁCH QEN VÀ KHÁCH MỚI
+----------+--------------+--------------+---------------+
|guest_type|total_bookings|total_canceled|cancel_rate_pct|
+----------+--------------+--------------+---------------+
|Khach moi |115580        |43672         |37.79          |
|Khach quen|3810          |552           |14.49          |
+----------+--------------+--------------+---------------+



In [13]:
result_4 = spark.sql("""
    WITH lead_time_groups AS (
        SELECT
            *,
            CASE
                WHEN lead_time > 90 THEN 'Dat truoc >90 ngay'
                WHEN lead_time < 7  THEN 'Dat sat ngay (<7 ngay)'
                ELSE                     'Khac'
            END AS lead_time_group
        FROM hotel
    )
    SELECT
        lead_time_group,
        COUNT(*)                                                                    AS total_bookings,
        SUM(CASE WHEN is_canceled = 0 THEN 1 ELSE 0 END)                           AS successful_checkins,
        ROUND(SUM(CASE WHEN is_canceled = 0 THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS checkin_success_rate_pct
    FROM lead_time_groups
    GROUP BY lead_time_group
    ORDER BY total_bookings DESC
""")

print("CÂU 4: TÁC ĐỘNG CỦA THỜI GIAN ĐẶT PHÒNG TRƯỚC (LEAD TIME)")
result_4.show(truncate=False)

CÂU 4: TÁC ĐỘNG CỦA THỜI GIAN ĐẶT PHÒNG TRƯỚC (LEAD TIME)
+----------------------+--------------+-------------------+------------------------+
|lead_time_group       |total_bookings|successful_checkins|checkin_success_rate_pct|
+----------------------+--------------+-------------------+------------------------+
|Dat truoc >90 ngay    |51131         |25233              |49.35                   |
|Khac                  |49844         |33248              |66.7                    |
|Dat sat ngay (<7 ngay)|18415         |16685              |90.61                   |
+----------------------+--------------+-------------------+------------------------+



In [14]:
result_5 = spark.sql("""
    WITH channel_stats AS (
        SELECT
            market_segment,
            COUNT(*)                                        AS total_bookings,
            ROUND(AVG(adr), 2)                              AS avg_adr,
            SUM(is_canceled)                                AS total_canceled,
            ROUND(SUM(is_canceled) / COUNT(*) * 100, 2)     AS cancel_rate_pct
        FROM hotel
        GROUP BY market_segment
        HAVING COUNT(*) >= 100
    )
    SELECT *
    FROM channel_stats
    ORDER BY avg_adr DESC
""")

print("CÂU 5: RỦI RO TÀI CHÍNH THEO KÊNH PHÂN PHỐI (>= 100 booking)")
result_5.show(truncate=False)

CÂU 5: RỦI RO TÀI CHÍNH THEO KÊNH PHÂN PHỐI (>= 100 booking)
+--------------+--------------+-------+--------------+---------------+
|market_segment|total_bookings|avg_adr|total_canceled|cancel_rate_pct|
+--------------+--------------+-------+--------------+---------------+
|Online TA     |56477         |117.2  |20739         |36.72          |
|Direct        |12606         |115.45 |1934          |15.34          |
|Aviation      |237           |100.14 |52            |21.94          |
|Offline TA/TO |24219         |87.35  |8311          |34.32          |
|Groups        |19811         |79.48  |12097         |61.06          |
|Corporate     |5295          |69.36  |992           |18.73          |
|Complementary |743           |2.89   |97            |13.06          |
+--------------+--------------+-------+--------------+---------------+



In [15]:
result_6 = spark.sql("""
    SELECT
        reservation_status_date,
        ROUND(AVG(adr), 2)                                      AS daily_avg_adr,
        ROUND(
            AVG(AVG(adr)) OVER (
                ORDER BY reservation_status_date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ), 2
        )                                                       AS rolling_7day_avg_adr
    FROM hotel
    GROUP BY reservation_status_date
    ORDER BY reservation_status_date ASC
""")

print("CÂU 6: GIÁ PHÒNG TRUNG BÌNH TRƯỢT 7 NGÀY")
result_6.show(30, truncate=False)

CÂU 6: GIÁ PHÒNG TRUNG BÌNH TRƯỢT 7 NGÀY
+-----------------------+-------------+--------------------+
|reservation_status_date|daily_avg_adr|rolling_7day_avg_adr|
+-----------------------+-------------+--------------------+
|2014-10-17             |62.8         |62.8                |
|2014-11-18             |0.0          |31.4                |
|2015-01-01             |62.06        |41.62               |
|2015-01-02             |9.63         |33.62               |
|2015-01-18             |0.0          |26.9                |
|2015-01-20             |76.5         |35.17               |
|2015-01-21             |37.3         |35.47               |
|2015-01-22             |116.57       |43.15               |
|2015-01-28             |244.0        |78.01               |
|2015-01-29             |66.0         |78.57               |
|2015-01-30             |56.81        |85.31               |
|2015-02-02             |79.74        |96.7                |
|2015-02-05             |155.83       |108.0

In [16]:
spark.stop()
print("SparkSession dừng")

SparkSession dừng
